# 02 · AutoDL 迁移与限额 canary

**先核实显卡名称。** Quadro P6000 是 24GB；这里的 96GB 目标通常是 RTX PRO 6000 Blackwell。必须以 `nvidia-smi`、计算能力、实际可用显存为准，不能仅按租赁商品简称判断。

本册先做文件完整性和 GPU 小算子检查，再规划受限 canary。G1 PASS 只意味着值得进行 canary；G2 是 canary 科学门槛，G3 是完整矩阵准备门槛。必须 G3 PASS 才允许完整矩阵。PyTorch GEMM 能运行不代表 fast-hadamard-transform、GPTQ 或全模型图已经兼容。

**迁移保留：** 源码、配置、锁定版本、随机种子、旋转/优化器状态、数据/token 哈希、日志与指标。**重新构建：** 依赖 CUDA/SM 的扩展；不要直接复制另一 GPU 的编译缓存。

## 路径与环境配置

CPU 默认路径不下载模型、不调用付费服务、不自动升级 PyTorch。**只上传 ipynb 不够，必须一起提供 src/configs/docs。** 首次在 Colab 上传整个 ZIP（或放 Drive），设置 `KIT_ARCHIVE`，将 `EXTRACT_ARCHIVE=True`，即可用下面单元安全解压；Drive 路径先设置 `MOUNT_DRIVE=True`。如果已经手动解压，则直接设置 `PROJECT_ROOT`。缺少基础 Python 依赖时才把 `INSTALL_BASE_DEPENDENCIES=True`。Colab 断线会丢失 `/content`，需要在阶段完成后同步结果；高频权重与激活 I/O 应留在本地磁盘。

访问受限 LLaMA 权重需要你已有的 Hugging Face 访问权。凭据只通过环境变量或 Colab Secrets 提供，不能写入 notebook、配置、提示词、日志或 Git。

In [1]:
from pathlib import Path
import os, sys, json

# Colab: 解压整个项目到 /content/cellquant_research_kit；不要仅上传 ipynb。
# AutoDL: 建议 /root/autodl-tmp/cellquant_research_kit。
# 留 None 时依次查找当前目录、父目录和上述两个位置。
PROJECT_ROOT = os.environ.get("CELLQUANT_PROJECT_ROOT") or None
RUN_ROOT = os.environ.get("CELLQUANT_RUN_ROOT") or None
KIT_ARCHIVE = None  # 例：/content/drive/MyDrive/cellquant_research_kit.zip
EXTRACT_ARCHIVE = False  # 首次运行且 ZIP 已上传时才改 True
ARCHIVE_DESTINATION = "/content/cellquant_workspace" if Path("/content").exists() else str(Path.cwd().parent / "cellquant_workspace")
MOUNT_DRIVE = False
DURABLE_ROOT = None  # 例：/content/drive/MyDrive/cellquant_runs；不存权重或 token
INSTALL_BASE_DEPENDENCIES = False
SEED = 20261115

In [2]:
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Drive mounting skipped; outputs remain under RUN_ROOT until copied.")

Drive mounting skipped; outputs remain under RUN_ROOT until copied.


In [3]:
archive_roots = []
if EXTRACT_ARCHIVE:
    import zipfile, hashlib
    if not KIT_ARCHIVE or not Path(KIT_ARCHIVE).is_file():
        raise FileNotFoundError("先上传完整 ZIP 并设置 KIT_ARCHIVE")
    archive_path = Path(KIT_ARCHIVE)
    destination = Path(ARCHIVE_DESTINATION).resolve()
    digest = hashlib.sha256(archive_path.read_bytes()).hexdigest()
    receipt = destination / "archive_source_sha256.txt"
    if destination.exists() and any(destination.iterdir()):
        if not receipt.is_file() or receipt.read_text().strip() != digest:
            raise FileExistsError("目标目录已有不同内容；请设置新的 ARCHIVE_DESTINATION，避免覆盖修改")
        print("Same archive already extracted; preserving existing project edits.")
    else:
        with zipfile.ZipFile(archive_path) as archive:
            infos = archive.infolist()
            if sum(info.file_size for info in infos) > 2 * 2**30:
                raise ValueError("项目包解压体积超过 2GiB；请勿把基础权重放进代码包")
            for info in infos:
                target = (destination / info.filename).resolve()
                if not target.is_relative_to(destination) or (info.external_attr >> 16) & 0o170000 == 0o120000:
                    raise ValueError("Unsafe archive member")
            destination.mkdir(parents=True, exist_ok=True)
            archive.extractall(destination)
        receipt.write_text(digest + "\n")
    archive_roots = [destination, destination / "cellquant_research_kit"]

candidates = ([Path(PROJECT_ROOT)] if PROJECT_ROOT else []) + archive_roots + [
    Path.cwd(), Path.cwd().parent,
    Path("/content/cellquant_research_kit"),
    Path("/root/autodl-tmp/cellquant_research_kit"),
]
roots = [p.resolve() for p in candidates
         if (p / "pyproject.toml").is_file() and (p / "src/cellquant").is_dir()]
if not roots:
    raise FileNotFoundError("请先解压完整项目包，并设置 PROJECT_ROOT；只上传 notebook 不足以运行。")
PROJECT_ROOT = roots[0]
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.environ["CELLQUANT_PROJECT_ROOT"] = str(PROJECT_ROOT)

if INSTALL_BASE_DEPENDENCIES:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r",
                           str(PROJECT_ROOT / "requirements/base.txt")])

from cellquant.runtime import (atomic_json, environment_report, resolve_paths,
                              build_manifest, verify_manifest, copy_completed_run)
PATHS = resolve_paths(PROJECT_ROOT, RUN_ROOT, DURABLE_ROOT)
print("Project:", PATHS["project"])
print("Runs:", PATHS["runs"])

Project: /workspace/scratch/506a2af5b817/cellquant_research_kit
Runs: /workspace/scratch/506a2af5b817/cellquant_research_kit/runs


## 1. 从 Colab 复制完成的结果

项目 ZIP 先解压到 AutoDL 本地数据盘。模型缓存和大激活文件应放本地 SSD。下面只复制可审查结果文件；基础模型应使用相同 checkpoint revision，从授权缓存或重新下载获得，不在项目包里传播权重。

In [4]:
SOURCE_RUN_DIR = None  # 例如已经上传的 /root/autodl-tmp/colab_runs/01_real_layer_probe
COPY_MIGRATED_RUN = False
OUT = PATHS["runs"] / "02_migration"
OUT.mkdir(parents=True, exist_ok=True)
if COPY_MIGRATED_RUN:
    if not SOURCE_RUN_DIR:
        raise ValueError("Set SOURCE_RUN_DIR first")
    copied = copy_completed_run(SOURCE_RUN_DIR, OUT / "imported_run")
    manifest = json.loads(Path(copied["manifest"]).read_text())
    integrity = verify_manifest(OUT / "imported_run", manifest)
    print(integrity)
    if integrity["status"] != "PASS":
        raise RuntimeError("迁移工件校验失败")
else:
    print("Migration copy skipped; this is not evidence that a prior run was transferred.")

Migration copy skipped; this is not evidence that a prior run was transferred.


## 2. 实际设备与小算子 smoke

本单元在没有 CUDA 的机器上会明确 SKIPPED。只有在目标 96GB GPU 上执行并保存结果，才能用于迁移记录。GEMM/FHT smoke 仍不是完整 G2；后面还需同面板旋转一致性、全精度模型等价、量化和 canary 评估。

In [5]:
from cellquant.runtime import gpu_kernel_probe
gpu_report = gpu_kernel_probe(test_optional_fht=True)
atomic_json(OUT / "gpu_kernel_probe.json", gpu_report)
print(json.dumps(gpu_report, ensure_ascii=False, indent=2))

{
  "status": "SKIPPED",
  "environment": {
    "schema_version": "1.0",
    "created_at_utc": "2026-11-15T09:43:10.754056+00:00",
    "python": "3.12.14",
    "executable": "/opt/codex/runtimes/codex-primary-runtime/dependencies/python/bin/python",
    "platform": "Linux-6.18.57-x86_64-with-glibc2.41",
    "machine": "x86_64",
    "packages": {
      "numpy": "2.3.5",
      "scipy": "1.17.0",
      "torch": null,
      "transformers": null,
      "datasets": null,
      "accelerate": null,
      "safetensors": null,
      "nbformat": "5.11.1",
      "nbclient": "0.11.0",
      "fast-hadamard-transform": null,
      "flash-attn": null,
      "triton": null
    },
    "cuda": {
      "available": false,
      "status": "SKIPPED_TORCH_NOT_INSTALLED"
    },
    "compatibility_claim": "CPU tooling only; GPU kernels require target-device tests"
  },
  "reason": "No CUDA device; target GPU compatibility remains untested"
}


## 3. 创建受限 canary 清单

运行需要真实 G1 PASS 及其原始证据哈希，且 Codex 已实现 `cellquant.experiment`。`RUN_CANARY=True` 只是执行开关，不能代替科学门槛。此处不会伪造后端或绕过缺失证据。

调试环境不修改完整实验协议。任何 CUDA/PyTorch/量化 kernel 变化都必须重新检查等价性和关键指标，并保存新的环境锁定文件。

In [6]:
from cellquant.runner import make_job_manifest, check_execution_readiness, execute_manifest
cfg_path = PROJECT_ROOT / "configs/autodl_canary.json"
canary_cfg = json.loads(cfg_path.read_text()) if cfg_path.exists() else {"jobs":[], "gpu_hour_budget":8}
plan = make_job_manifest(canary_cfg.get("jobs", []), project_root=PROJECT_ROOT,
                         gpu_hour_budget=canary_cfg.get("gpu_hour_budget", 8), required_gate="G1",
                         execution_provenance_path=canary_cfg.get("execution_provenance_path"))
atomic_json(OUT / "canary_manifest.json", plan)
GATE_PATH = PATHS["runs"] / "gates/G1.json"
EVIDENCE_PATH = PATHS["runs"] / "gates/G1_evidence.json"
readiness = check_execution_readiness(plan, gate_path=GATE_PATH, gate_evidence_path=EVIDENCE_PATH)
print("Planned jobs:", len(plan["jobs"]))
print(json.dumps(readiness, ensure_ascii=False, indent=2))

Planned jobs: 6
{
  "status": "BLOCKED",
  "missing": [
    "Experiment backend missing callable run_experiment / BACKEND_CONTRACT_VERSION='1.0'",
    "Verified model/tokenizer/data/protocol provenance file missing",
    "Manifest was planned without verified execution provenance",
    "Required gate report is missing"
  ],
  "scope": "Execution prerequisites only; the gate evaluator owns scientific acceptance"
}


In [7]:
RUN_CANARY = False
canary_execution = execute_manifest(
    plan, project_root=PROJECT_ROOT, gate_path=GATE_PATH,
    gate_evidence_path=EVIDENCE_PATH, state_dir=OUT / "canary_execution",
    allow_execution=RUN_CANARY,
)
print(json.dumps(canary_execution, ensure_ascii=False, indent=2))

{
  "status": "MANIFEST_ONLY",
  "readiness": {
    "status": "BLOCKED",
    "missing": [
      "Experiment backend missing callable run_experiment / BACKEND_CONTRACT_VERSION='1.0'",
      "Verified model/tokenizer/data/protocol provenance file missing",
      "Manifest was planned without verified execution provenance",
      "Required gate report is missing"
    ],
    "scope": "Execution prerequisites only; the gate evaluator owns scientific acceptance"
  },
  "jobs_executed": 0
}


## 4. 迁移完成的验收标准

1. 工件 SHA、模型 revision、token 样本哈希相符；不要只看文件名。
2. 目标 GPU 上 GEMM/FHT、旋转融合、量化前等价和 GPTQ 都实际通过。
3. 校准与评估峰值显存有安全余量；完整记录实际 `allocated` 和 `reserved`。
4. 原平台和目标平台的同面板/同模型结果差异在协议容差内；不同 GPU 不要求逐 bit 一致。
5. 完成 G2 canary 的质量、成本和强基线核查，并通过 G3 的恢复/协议/硬件验收后，才进入 03。

G3 证据还必须保存 `provenance.execution_fingerprint_sha256`，等于当前 manifest 的 `execution_fingerprint.sha256`。目标 GPU、driver、依赖、源码、模型或数据变化时，不能直接复用旧 G3 PASS；必须重新测量受影响项目并重新生成门槛报告。G1 可以跨机器用于进入 canary。

如果此 notebook 只输出 BLOCKED/SKIPPED，说明没有迁移或全模型成功证据，不是“自动修复成功”。